In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
import matplotlib.animation as animation
import sunpy.map
import astropy.units as u
from scipy.stats import truncnorm
import os
from datetime import datetime
import re
import Sunlimb
import image as img
from matplotlib.colors import Normalize, LogNorm
%matplotlib notebook

## 用于生成随机正弦信号

In [ ]:
def gaussian(dx, mean, std):
    dx = np.array(dx)
    return mean * np.exp(-dx ** 2 / (2 * std ** 2))

In [ ]:
def generate_signal(amplitude, omega, std_signal, mean_signal, std_noise, shape=(51, 51), seed=None):
    if seed is not None:
        np.random.seed(seed)
    noise = np.random.normal(mean_signal / 3, std_noise, shape)
    len_t, len_x = shape
    t_vect = np.linspace(0, len_t - 1, len_t).astype(int)
    x_vect = np.floor(
        len_x // 2 + amplitude * np.sin(omega * t_vect + np.random.uniform(0, 2 * np.pi)) + np.random.normal(0,
                                                                                                             amplitude / 4,
                                                                                                             len_t)).astype(
        int)
    signal = mean_signal * np.exp(-(np.arange(0, len_x)[None, :] - x_vect[:, None]) ** 2 / (2 * std_signal ** 2))
    signal += noise
    return signal

In [ ]:
signal = generate_signal(5, 0.08 * np.pi, 1, 10, 1, seed=100)
fig, ax = plt.subplots()
ax.imshow(signal, cmap='gray', origin='lower')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('signal with period of 25')
fig.savefig('./fig/amplitude/signal.png', dpi=300)

## 进行小波变换测试

In [ ]:
def imshow_vedio(fig, data, cmap, **kwargs):
    data = np.array(data)
    ax = fig.axes[0]
    im = ax.imshow(data[0], cmap=cmap, origin='lower', vmin=np.min(data), vmax=np.max(data))
    ax.set_xlabel(kwargs.get('xlabel', 'x'))
    ax.set_ylabel(kwargs.get('ylabel', 'y'))
    ax.set_aspect('auto')
    title = ax.set_title(kwargs.get('title', 't=0'))
    plt.colorbar(im, ax=ax, label=kwargs.get('name_cbar', None))

    def update(frame):
        polt_data = data[frame]
        im.set_data(polt_data)
        title.set_text(kwargs.get('title', f't')+f'={frame}')
        return [im, title]

    ani = animation.FuncAnimation(fig=fig, func=update, frames=data.shape[0], interval=kwargs.get('interval', 100))
    return ani

复数Morlet小波在pywt中的写法为：'cmorB-C‘，其中B和C是浮点数，分别表示小波的带宽频率和中心频率。尺度a，带宽频率B，中心频率C的复数Morlet小波的数学形式为：
$$\psi(t,a,B,C)=\frac{1}{\sqrt{a}}\pi^{-\frac{1}{4}}e^{i2\pi C\frac{t}{a}}e^{-\frac{1}{B}\left(\frac{t}{a}\right)^2}$$
不妨先假定波的模式为：
$$x=A\sin\left(k_t t+\phi\right)$$
故可以只对信号在t方向上做连续小波变换，得到$(x,t,scale_t)$处的振幅或者相位。
模拟信号的周期为25，故对于'cmor1.0-1.0'的复数Morlet小波，取尺度为25

In [ ]:
phase = []
amplitude = []
scales = np.arange(1, 64)
for i in range(signal.shape[-1]):
    coe, _ = pywt.cwt(signal[:, i], scales=scales, wavelet='cmor1.0-1.0')
    phase.append(np.angle(coe))
    amplitude.append(np.abs(coe))

在不同x处的小波振幅谱

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ani = imshow_vedio(fig, amplitude, cmap='jet', xlabel='t', ylabel='scale', title='x', name_cbar='amplititude', interval=500)
ani.save('./fig/amplitude/amplitude.gif', fps=4)

对尺度为25的小波的振幅谱作图

In [ ]:
amplitude_25 = np.array(amplitude)[:, 25, :].T
phase_25 = np.array(phase)[:, 25, :].T
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

im0 = axs[0].imshow(amplitude_25, cmap='jet', origin='lower')
axs[0].set_xlabel('x')
axs[0].set_ylabel('t')
axs[0].set_title('amplitude at scale 25')
plt.colorbar(im0, ax=axs[0], label='amplitude')

im1 = axs[1].imshow(phase_25/np.pi, cmap='hsv', origin='lower')
axs[1].set_xlabel('x')
axs[1].set_ylabel('t')
axs[1].set_title('phase/pi at scale 25')
plt.colorbar(im1, ax=axs[1], label='phase/pi')

fig.savefig('./fig/amplitude/amplitude_phase_25.png', dpi=300)

# 使用蒙特卡洛模拟生成一些信号

In [ ]:
import random
def simulation(x, t, dx=120, dt=3, num=200, seed=None):
    if seed is not None:
        np.random.seed(seed)

    life_span = {'mean': 100, 'std': 50} # s
    brightness = {'mean': 24, 'std': 8}
    period = {'min': 60, 'max': 120} # s
    amplitude = {'mean': 15, 'std': 3} # km/s
    width = 7
    noise = {'mean': 7, 'std': 17}

    x_vect = np.arange(0, x, dx)
    t_vect = np.arange(0, t, dt)

    space_time = np.zeros((len(t_vect), len(x_vect)))

    for i in range(num):
        begin_time = np.random.uniform(0, t)
        begin_x = np.random.uniform(0, x)
        life = truncnorm.rvs(-2, 3, loc=life_span['mean'], scale=life_span['std'])
        phase = np.random.uniform(0, 2*np.pi)
        T = np.random.uniform(period['min'], period['max'])
        I = truncnorm.rvs(-4, 4, loc=brightness['mean'], scale=brightness['std'])
        speed = truncnorm.rvs(-3, 3, loc=amplitude['mean'], scale=amplitude['std'])
        A = 0.5*np.cos(np.random.uniform(0, 0.5*np.pi))*speed*T/np.pi

        t_index = (np.arange(begin_time, begin_time+life, dt)/dt).astype(int)
        t_index = np.clip(t_index, 0, len(t_vect)-1)
        c_index = np.floor((begin_x + A*np.sin(2*np.pi * t_index*dt/T + phase) + np.random.normal(0, A/5, len(t_index)))/dx).astype(int)
        c_index = np.clip(c_index, 0, len(x_vect)-1)

        for j, time in enumerate(t_index):
            c = c_index[j]
            x_index = (np.array(range(width)) - width//2 + c).astype(int)
            x_index = np.clip(x_index, 0, len(x_vect)-1)
            for index in x_index:
                # space_time[time, index] = I * np.exp(-(index-c)**2 / (2 * (width/10)**2))
                space_time[time, index] = I * np.cos(np.pi/(2*(width//2))*(index-c))

    space_time += truncnorm.rvs(-5, 5, loc=noise['mean'], scale=noise['std'], size=(len(t_vect), len(x_vect)))
    return space_time

In [ ]:
def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1]+coe.data[2]
    # hq = img.box_car_smoothing(hq, kernel_size=5)
    hq[np.isnan(image)] = np.nan
    return hq

In [ ]:
st_data = simulation(301*120, 603, num=100, seed=0)
st_data = atrous(st_data)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(st_data, extent=[0, 0.12*301, 0, 603], cmap='sdoaia171', origin='lower')
plt.colorbar(im, ax=ax)
ax.set_xlabel('x/Mm')
ax.set_ylabel('t/s')
ax.set_aspect('auto')
ax.set_title('space time plot simulation with 100 waves')
fig.savefig('./fig/amplitude/space_time_simulation.png', dpi=400)

## 对生成的模拟信号进行小波变换测试

In [ ]:
phase = []
amplitude = []
scales = np.arange(1, 40)
for i in range(st_data.shape[-1]):
    coe, _ = pywt.cwt(st_data[:, i], scales=scales, wavelet='cmor1.0-1.0')
    phase.append(np.angle(coe))
    amplitude.append(np.abs(coe))

amplitude = np.array(amplitude)
phase = np.array(phase)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(amplitude[:, 15, :].T, cmap='jet', origin='lower', extent=[0, 0.12*301, 0, 603])
ax.set_xlabel('x/Mm')
ax.set_ylabel('t/s')
ax.set_aspect('auto')
ax.set_title('amplitude at scale 15(T=45s)')
plt.colorbar(im, ax=ax, label='amplitude')
fig.savefig('./fig/amplitude/amplitude at scale 15(T=45s).png', dpi=400)

# 对真实时空图进行测试

In [ ]:
file_path = "E:/python/projects/alfven/data/corrected/10moving/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)[295:495]

In [ ]:
import sunkit_image.enhance as enhance

def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1]+coe.data[2]
    # hq = img.box_car_smoothing(hq, kernel_size=7)
    hq[np.isnan(image)] = np.nan
    return hq

def unsharp_mask(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    unsharp_mask_image = img.unsharp_mask(image_nan_to_0, kernel_size=21, high_freq_only=True)
    unsharp_mask_image[np.isnan(image)] = np.nan
    return unsharp_mask_image

def wow(image):
    enhance.wow(image, denoise_coefficients=[5, 2, 1], soft_threshold=True, bilateral=1, gamma=0.7)

In [ ]:
deg_range = (174.5, 177.5)
d_deg = 0.01
d_time = 3
deg_vect = np.arange(deg_range[0], deg_range[1] + d_deg / 2, d_deg)
time_vect = np.arange(0, len(files) * d_time + 0.5 * d_time, d_time) / 60  # unit: minute
r = 7
fig_st, ax_st = plt.subplots(figsize=(6, 4))

st_data = Sunlimb.space_time_plot(ax=ax_st, files=files, deg_range=deg_range, r=r, d_time=3, d_deg=0.01, process=None,
                                  cmap='sdoaia171',
                                  title='space_time_plot r=7Mm', xlabel='degree', ylabel='time(min)', norm=Normalize(),
                                  colorbar=True)

进行unsharp_mask处理，提取特征

In [ ]:
sharpen_data = img.unsharp_mask(st_data, kernel_size=13, high_freq_only=True)
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(sharpen_data, cmap='sdoaia171', origin='lower', extent=[deg_vect[0], deg_vect[-1], time_vect[0], time_vect[-1]])
ax.set_aspect('auto')
ax.set_xlabel('degree')
ax.set_ylabel('time(min)')
ax.set_title('atrous & unsharp mask at 7Mm')
fig.savefig('./fig/amplitude/atrous_sharp_mask.png', dpi=400)

In [ ]:
phase = []
amplitude = []
scales = np.arange(1, 100)
for i in range(sharpen_data.shape[-1]):
    coe, _ = pywt.cwt(sharpen_data[:, i], scales=scales, wavelet='cmor1.0-1.0')
    phase.append(np.angle(coe))
    amplitude.append(np.abs(coe))

amplitude = np.array(amplitude)
phase = np.array(phase)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(amplitude[:, 0, :].T, cmap='jet', extent=[deg_vect[0], deg_vect[-1], time_vect[0], time_vect[-1]],
               origin='lower', vmin=amplitude.min(), vmax=amplitude.max())
ax.set_xlabel('degree')
ax.set_ylabel('time')
ax.set_aspect('auto')
title = ax.set_title('amplitude at scale 1(T=3s)')
plt.colorbar(im, ax=ax, label='amplitude')

def update(frame):
    plot_data = amplitude[:, frame, :].T
    im.set_data(plot_data)
    title.set_text(f'amplitude at scale {frame+1}(T={3*(frame+1)}s)')

ani = animation.FuncAnimation(fig=fig, func=update, frames=amplitude.shape[1], interval=500)
# ani.save('./movie/amplitude.gif', dpi=150)